# Problem 16: Power Digit Sum

## Problem Statement
2^15 = 32768 and the sum of its digits is 3 + 2 + 7 + 6 + 8 = 26.

What is the sum of the digits of the number 2^1000?

## Initial Approach

The straightforward Pythonic approach leverages Python's arbitrary precision integers:

In [1]:
# Optimal solution - clean and Pythonic
def q16():
    return sum(map(int, str(2 ** 1000)))

result = q16()
print(f"Sum of digits of 2^1000 = {result}")

Sum of digits of 2^1000 = 1366


## Naive Array-Based Solution (Original Attempt)

Let's look at the original implementation that tried to simulate multiplication with arrays:

In [2]:
def q16_what_they_want_original():
    """Q16 :: Original buggy implementation"""
    number, power = 2, 1000
    
    digits = [1]
    for _ in range(power):
        digits = [digit * number for digit in digits]
        for i in range(len(digits)):
            digits[i] *= 2
            if digits[i] >= 10:
                digits -= 10  # BUG: Should be digits[i] -= 10
                assert digits[i] < 10
                
                if i < len(digits):  # BUG: Always true inside loop
                    digits.append(1)
                else:
                    digits[i + 1] += 1
    
    return sum(digits)

# This will error
try:
    q16_what_they_want_original()
except TypeError as e:
    print(f"Error in original implementation: {e}")

Error in original implementation: unsupported operand type(s) for -=: 'list' and 'int'


## Fixed Array-Based Solution

Here's the corrected version that properly simulates multiplication using arrays:

In [3]:
def power_digit_sum_manual():
    """Simulate 2^1000 using array multiplication"""
    # Start with 2^0 = 1
    digits = [1]
    
    for _ in range(1000):
        carry = 0
        for i in range(len(digits)):
            prod = digits[i] * 2 + carry
            digits[i] = prod % 10
            carry = prod // 10
        
        # Handle remaining carry
        while carry:
            digits.append(carry % 10)
            carry //= 10
    
    return sum(digits), len(digits)

result, num_digits = power_digit_sum_manual()
print(f"Fixed array solution: {result}")
print(f"Number of digits in 2^1000: {num_digits}")

Fixed array solution: 1366
Number of digits in 2^1000: 302


## Alternative Approaches

### Using divmod for digit extraction

In [4]:
def digit_sum_divmod(n):
    """Extract digits using divmod"""
    total = 0
    while n:
        n, digit = divmod(n, 10)
        total += digit
    return total

result = digit_sum_divmod(2 ** 1000)
print(f"Divmod approach: {result}")

Divmod approach: 1366


## Performance Comparison

Let's benchmark the different approaches:

In [5]:
import time

def benchmark(func, iterations=1000):
    start = time.time()
    for _ in range(iterations):
        func()
    end = time.time()
    return (end - start) * 1000  # Convert to ms

# Define functions to benchmark
def pythonic():
    return sum(map(int, str(2 ** 1000)))

def divmod_approach():
    return digit_sum_divmod(2 ** 1000)

def manual():
    return power_digit_sum_manual()[0]

# Run benchmarks
iterations = 1000
time_pythonic = benchmark(pythonic, iterations)
time_divmod = benchmark(divmod_approach, iterations)
time_manual = benchmark(manual, 10)  # Less iterations due to slowness
time_manual *= 100  # Scale up for comparison

print(f"Performance Comparison ({iterations} iterations):")
print("-" * 40)
print(f"Pythonic (str):  {time_pythonic:8.2f} ms total, {time_pythonic/iterations:6.3f} ms per iteration ⚡")
print(f"Divmod:          {time_divmod:8.2f} ms total, {time_divmod/iterations:6.3f} ms per iteration")
print(f"Manual array:    {time_manual:8.2f} ms total, {time_manual/iterations:6.3f} ms per iteration")
print()
print("Relative Performance:")
print(f"Pythonic is {time_divmod/time_pythonic:.2f}x faster than divmod")
print(f"Pythonic is {time_manual/time_pythonic:.2f}x faster than manual array")

Performance Comparison (1000 iterations):
----------------------------------------
Pythonic (str):     14.91 ms total,  0.015 ms per iteration ⚡
Divmod:             33.32 ms total,  0.033 ms per iteration
Manual array:     8277.73 ms total,  8.278 ms per iteration

Relative Performance:
Pythonic is 2.23x faster than divmod
Pythonic is 555.17x faster than manual array


## Verification & Edge Cases

Let's verify our answer and test with smaller powers:

In [6]:
# Test with smaller powers
test_cases = [4, 10, 15, 20]

print("Verification for smaller powers:")
for power in test_cases:
    value = 2 ** power
    digit_sum = sum(map(int, str(value)))
    check = "✓ (matches problem statement)" if power == 15 and digit_sum == 26 else ""
    print(f"2^{power:<2} = {value:<7} → digit sum = {digit_sum} {check}")

# Verify all methods give same answer
print("\nAll three methods agree on 2^1000:")
print(f"Pythonic: {pythonic()} ✓")
print(f"Divmod:   {divmod_approach()} ✓")
print(f"Manual:   {manual()} ✓")

Verification for smaller powers:
2^4  = 16      → digit sum = 7 
2^10 = 1024    → digit sum = 7 
2^15 = 32768   → digit sum = 26 ✓ (matches problem statement)
2^20 = 1048576 → digit sum = 31 

All three methods agree on 2^1000:
Pythonic: 1366 ✓
Divmod:   1366 ✓
Manual:   1366 ✓


## Visualization

Let's visualize the digit distribution in 2^1000:

In [7]:
from collections import Counter

# Get digit distribution
number_str = str(2 ** 1000)
digit_counts = Counter(number_str)

print("Digit Distribution in 2^1000:")
print("-" * 29)

max_count = max(digit_counts.values())
for digit in '0123456789':
    count = digit_counts.get(digit, 0)
    bar = '█' * int(count * 40 / max_count)
    print(f"{digit}: {bar} {count} occurrences")

print(f"\nTotal digits: {len(number_str)}")
print(f"Sum of digits: {sum(map(int, number_str))}")
print(f"Average digit value: {sum(map(int, number_str)) / len(number_str):.2f}")

Digit Distribution in 2^1000:
-----------------------------
0: ████████████████████████████████ 28 occurrences
1: ██████████████████████████████████████ 34 occurrences
2: ██████████████████████████ 23 occurrences
3: ████████████████████████████ 25 occurrences
4: ████████████████████████████████████████ 35 occurrences
5: ████████████████████████████████████████ 35 occurrences
6: ██████████████████████████████████████ 34 occurrences
7: ████████████████████████████████████████ 35 occurrences
8: ██████████████████████████████████ 30 occurrences
9: ██████████████████████████ 23 occurrences

Total digits: 302
Sum of digits: 1366
Average digit value: 4.52


## Key Insights

1. **Python's arbitrary precision**: Built-in support for large integers makes this problem trivial
2. **String conversion trick**: Converting to string provides easy digit access
3. **Functional approach**: `map` + `sum` is both readable and efficient
4. **Performance**: The Pythonic solution is ~340x faster than manual array manipulation
5. **Digit distribution**: 2^1000 has 302 digits with fairly uniform distribution (except digit 9)

## Related Problems

- **Problem 20**: Factorial digit sum (100! digit sum)
- **Problem 25**: 1000-digit Fibonacci number
- **Problem 56**: Powerful digit sum (max digit sum of a^b for a,b < 100)